# AWG Hardware Connection Test — Spectrum M4i.6631-x8

Purpose: validate the physical connection to the Spectrum Instrumentation AWG/DDS
card before running the full `atommovr` / `awg_controller` rearrangement pipeline.
Every section past Section 1 turns on real RF output — read the safety guidelines
below before running anything.

## Safety guidelines (read before running any generation cell)

- **Hard ceiling: output amplitude must never exceed 2.0 V.** This is enforced in
  code (`MAX_SAFE_OUTPUT_V` in `awg_controller/src/scapp.py`, and the assertion in
  `atommover_controller.py`'s `_initialize_scapp`) — do not bypass it. Exceeding it
  can **permanently damage the AOD amplifier**.
- The manufacturer-recommended default is **1.6 V** (`HardwareConfig.max_amplitude_v`).
  **This notebook starts conservative at 1.0 V** in every section — raise it
  deliberately, in small steps, only after verifying each step on a scope.
- **Always verify the output on an oscilloscope before connecting a real AOD
  amplifier.** Do not connect the amplifier while running Sections 1–3 for the
  first time; only connect it once waveform shape/amplitude/frequency have been
  confirmed safe on a scope.
- Per-channel tone-amplitude sum must stay **≤ 40 % of full scale**
  (`MAX_AMPLITUDE_PCT_PER_CHANNEL` in `awg_controller/src/awg_control.py`). Every
  test below uses a single tone at the full 40 % per-tone budget — do not add
  extra simultaneous tones without re-deriving that budget.
- `spcm` (and `cupy`, for the SCAPP sections; `pyvisa`, for the spectrum-analyzer
  section) are optional dependencies. If any import fails, those cells cannot run
  in hardware mode — check the driver/CUDA/VISA installation rather than working
  around the import.
- Run cells **top to bottom, one at a time** — do not use "Run All". Confirm each
  section's result (console output / scope trace) before moving to the next,
  especially before Section 5 (experiment-timescale ramp).
- Always run the **Cleanup** cell (Section 6) at the end of a session, even after
  an error — it stops the feeder/card and analyzer connection and releases the
  DMA buffer. A card left streaming keeps driving the AOD.
- If any cell raises an exception, don't re-run blindly — run Section 6 (cleanup)
  first so the card is left in a known, stopped state.


# 1. Driver check

Opens the first available AWG-type card, prints identification info, and closes
again immediately. Channels/clock/trigger/amplitude are never touched here, so
this cell is safe to run even with an amplifier already connected.

In [ ]:
import spcm
from spcm import units

with spcm.Card(card_type=spcm.SPCM_TYPE_AO, verbose=True) as card:
    print(f"Serial number:    {card.sn()}")
    print(f"Function type:    {card.function_type()}")
    print(f"Max sample value: {card.max_sample_value()}")

    clock = spcm.Clock(card)
    max_rate = clock.sample_rate(max=True, return_unit=units.MHz)
    print(f"Max sample rate:  {max_rate}")

print("Driver check OK -- card opened, queried, and closed without errors.")


# 2. Spectrum analyzer connection (Siglent SVA1015X)

Connect to the Siglent SVA1015X spectrum analyzer over VISA so the ramp tests in
Sections 4–5 can be verified automatically instead of only by eye. Requires
`pyvisa` (`pip install pyvisa`); for the LAN resource string below you also need a
VISA backend — either NI-VISA/Keysight IO Libraries, or the pure-Python
`pyvisa-py` backend (`pip install pyvisa-py`).

This section is optional: skip it and the `if "sva" in globals()` guards further
down leave Sections 4–5 working scope-only, exactly as before.

**Double-check the exact SCPI mnemonics below against the official SVA1000X
remote-control manual** — they follow the instrument's documented command set but
haven't been verified against real hardware from this environment.

In [3]:
import pyvisa

rm = pyvisa.ResourceManager()
print(rm.list_resources())  # run once to find the exact VISA resource string below


('ASRL1::INSTR', 'ASRL2::INSTR', 'ASRL3::INSTR', 'ASRL4::INSTR', 'ASRL5::INSTR', 'USB0::0xF4EC::0x1301::SVA1XA1Q800517::INSTR')


Set `SVA_RESOURCE` to match your setup (uncomment the USB-TMC line instead if
that's how the analyzer is connected), then open it.

In [4]:
# SVA_RESOURCE = "TCPIP::192.168.1.100::inst0::INSTR"  # LAN (VXI-11) -- replace with the analyzer's IP
SVA_RESOURCE = "USB0::0xF4EC::0x1301::SVA1XA1Q800517::INSTR"  # USB-TMC -- get the exact string from rm.list_resources()

sva = rm.open_resource(SVA_RESOURCE)
sva.timeout = 5000  # ms
sva.read_termination = "\n"
sva.write_termination = "\n"

idn = sva.query("*IDN?").strip()
print(f"Connected to: {idn}")
assert "SVA1015X" in idn, f"Unexpected instrument reply: {idn!r}"


Connected to: Siglent Technologies,SVA1015X,SVA1XA1Q800517,3.2.2.6.0R10


In [7]:
def set_span(center_hz, span_hz):
    """Center the analyzer's sweep on center_hz with the given span (Hz)."""
    sva.write(f":SENSe:FREQuency:CENTer {center_hz}")
    sva.write(f":SENSe:FREQuency:SPAN {span_hz}")


def read_peak():
    """Move marker 1 to the highest peak in the current sweep and read it back
    as (freq_hz, amplitude_dbm).
    """
    sva.write(":CALCulate:MARKer1:STATe ON")
    sva.write(":CALCulate:MARKer1:MAXimum")
    freq_hz = float(sva.query(":CALCulate:MARKer1:X?"))
    amp_dbm = float(sva.query(":CALCulate:MARKer1:Y?"))
    return freq_hz, amp_dbm


set_span(center_hz=80e6, span_hz=80e6)  # covers the 60-100 MHz AOD tone band with margin
print("Spectrum analyzer ready.")


Spectrum analyzer ready.


# 3. Simple function generation

Bare-metal SCAPP test (no `awg_controller` abstraction yet): stream one static,
phase-continuous tone on channel 0 via `spcm.SCAPPTransfer` + CuPy, following the
same pattern as `spcm-examples/10_cuda_scapp/5_scapp_gen_fifo_sine.py`. This
confirms the GPU-direct RDMA generation path itself works before testing our
`ScappFeeder` wrapper in Sections 4–5.

**Before running:** amplifier disconnected or oscilloscope in place. Amplitude is
0.4 x full scale (the 40 % per-tone budget for a single tone) at a conservative
1.0 V channel range.

In [ ]:
import time

import cupy as cp
import spcm
from spcm import units

MAX_AMPLITUDE_V = 1.0       # conservative; hard ceiling is 2.0 V (see safety notes above)
TEST_FREQUENCY_HZ = 10e6    # 10 MHz static tone -- easy to confirm on a scope/spectrum analyzer
RUN_SECONDS = 5.0

assert MAX_AMPLITUDE_V <= 2.0, "exceeds hard safety ceiling -- see safety notes above"

with spcm.Card(card_type=spcm.SPCM_TYPE_AO, verbose=True) as card:
    card.card_mode(spcm.SPC_REP_FIFO_SINGLE)
    card.timeout(5 * units.s)

    trigger = spcm.Trigger(card)
    trigger.or_mask(spcm.SPC_TMASK_SOFTWARE)

    channels = spcm.Channels(card, card_enable=spcm.CHANNEL0)
    channels.enable(True)
    channels.output_load(50 * units.ohm)
    channels.amp(MAX_AMPLITUDE_V * units.V)

    clock = spcm.Clock(card)
    clock.mode(spcm.SPC_CM_INTPLL)
    sample_rate_hz = clock.sample_rate(max=True, return_unit=units.Hz).to_base_units().magnitude
    max_value = card.max_sample_value()
    print(f"Sample rate: {sample_rate_hz / 1e6:.1f} MHz")

    notify_samples = 512 * 1024
    dma_buffer_samples = 32 * 1024 * 1024

    scapp_transfer = spcm.SCAPPTransfer(card, direction=spcm.Direction.Generation)
    scapp_transfer.notify_samples(notify_samples)
    scapp_transfer.allocate_buffer(dma_buffer_samples)
    scapp_transfer.start_buffer_transfer(spcm.M2CMD_DATA_STARTDMA)
    cp_dtype = scapp_transfer.numpy_type()

    t_local = cp.arange(notify_samples, dtype=cp.float64) / sample_rate_hz
    phase_s = 0.0
    started = False
    t_start = time.monotonic()
    try:
        for card_buffer in scapp_transfer:
            card_buffer[0, :] = (
                cp.sin(2 * cp.pi * TEST_FREQUENCY_HZ * (t_local + phase_s)) * 0.4 * max_value
            ).astype(cp_dtype)
            phase_s += notify_samples / sample_rate_hz

            if not started and scapp_transfer.fill_size_promille() > 800:
                card.start(spcm.M2CMD_CARD_ENABLETRIGGER)
                started = True
                print("Card started -- check the scope/spectrum analyzer now.")

            if time.monotonic() - t_start > RUN_SECONDS:
                break
    finally:
        card.stop(spcm.M2CMD_DATA_STOPDMA | spcm.M2CMD_CARD_STOP)

print(f"Done -- static {TEST_FREQUENCY_HZ / 1e6:.0f} MHz tone should have appeared "
      f"on channel 0 for ~{RUN_SECONDS:.0f} s.")


# 4. Single Ramp testing (observable timescale)

Now test our own `ScappFeeder` (`awg_controller/src/scapp.py`) — the class the real
`atommover_controller` uses — rather than raw `spcm`/CuPy. One tone, one channel,
ramped slowly enough (a few seconds) to watch the frequency sweep by eye on a
scope/spectrum analyzer (or automatically, via the Siglent SVA1015X connection in
Section 2). This validates the feeder's setup + phase-continuous ramp math with a
duration far longer than one `notify_samples` window, so there's no risk of a
dropped transition (see Section 5 for that check).

This cell opens the card and starts the feeder; keep `card`/`feeder` alive for
Section 5, then release both in Section 6 (Cleanup).

In [ ]:
import spcm

from awg_controller.scripts.atommover_controller import HardwareConfig
from awg_controller.src.awg_control import AODSettings, AWGBatch, RFRamp
from awg_controller.src.scapp import ScappFeeder, ScappFeederConfig

hw = HardwareConfig(card_path="/dev/spcm0", max_amplitude_v=1.0)  # conservative; hard ceiling 2.0 V
aod = AODSettings(f_min_v=60e6, f_max_v=100e6, f_min_h=60e6, f_max_h=100e6, grid_rows=1, grid_cols=1)


def hold_at(freq_hz):
    """Single-tone holding batch at freq_hz, full 40% per-tone budget (only tone on this channel)."""
    return AWGBatch(
        ramps=[RFRamp(channel=0, core=0, f_start=freq_hz, f_end=freq_hz, amplitude_pct=40.0, tone_index=0)],
        total_duration_s=0.0,
    )


card = spcm.Card(hw.card_path)
feeder = ScappFeeder(card, hw, aod, ScappFeederConfig(ramp_shape="linear"))
feeder.start(hold_at(aod.f_min_v))
print(f"Feeder started -- sample_rate={feeder.sample_rate_hz / 1e6:.1f} MHz, "
      f"holding at {aod.f_min_v / 1e6:.0f} MHz.")


Run the observable ramp: sweep the single tone from `f_min_v` to `f_max_v` over a
few seconds. Watch the spectrum analyzer/scope while this cell runs.

In [ ]:
OBSERVABLE_RAMP_S = 3.0

ramp_batch = AWGBatch(
    ramps=[RFRamp(channel=0, core=0, f_start=aod.f_min_v, f_end=aod.f_max_v, amplitude_pct=40.0, tone_index=0)],
    total_duration_s=OBSERVABLE_RAMP_S,
)
print(f"Ramping {aod.f_min_v / 1e6:.0f} -> {aod.f_max_v / 1e6:.0f} MHz over "
      f"{OBSERVABLE_RAMP_S:.0f} s -- watch the spectrum analyzer/scope now.")
feeder.submit_batch(ramp_batch)     # blocks for ~OBSERVABLE_RAMP_S
feeder.submit_holding(hold_at(aod.f_max_v))
print(f"Ramp complete -- now holding at {aod.f_max_v / 1e6:.0f} MHz. "
      f"dropped_transition_count={feeder.dropped_transition_count}")

if "sva" in globals() and sva is not None:
    freq_hz, amp_dbm = read_peak()
    print(f"SVA1015X peak: {freq_hz / 1e6:.3f} MHz @ {amp_dbm:.1f} dBm "
          f"(expected ~{aod.f_max_v / 1e6:.0f} MHz)")


# 5. Single Ramp testing (experiment timescale)

Repeat the same single-tone ramp, but at a real rearrangement-move duration
instead of an observable one. `MIN_MOVE_DURATION_S` (`atommovr/utils/timing.py`) is
the floor `atommovr` uses for a single-site move; typical moves on the tutorial
lattice run ~3-40 us (see the `ScappFeederConfig` docstring in
`awg_controller/src/scapp.py`).

This is the functionally critical check: at this timescale, one `submit_batch`
call can land inside the same `notify_samples` DMA chunk as the previous one, in
which case the transition is silently overwritten before it ever reaches the DAC
(`ScappFeeder.dropped_transition_count`) -- exactly the failure mode the docstring
flags as unverified on real hardware. A single tone is the simplest possible case;
it does not exercise full-grid throughput, so a clean result here is necessary but
not sufficient to validate a real multi-tone rearrangement round.

In [ ]:
from atommovr.utils.timing import MIN_MOVE_DURATION_S

EXPERIMENT_RAMP_S = MIN_MOVE_DURATION_S  # 1 us floor; real moves run ~3-40 us

fast_batch = AWGBatch(
    ramps=[RFRamp(channel=0, core=0, f_start=aod.f_max_v, f_end=aod.f_min_v, amplitude_pct=40.0, tone_index=0)],
    total_duration_s=EXPERIMENT_RAMP_S,
)

dropped_before = feeder.dropped_transition_count
feeder.submit_batch(fast_batch)
feeder.submit_holding(hold_at(aod.f_min_v))
dropped_after = feeder.dropped_transition_count

print(f"Experiment-timescale ramp: {EXPERIMENT_RAMP_S * 1e6:.1f} us, "
      f"{aod.f_max_v / 1e6:.0f} -> {aod.f_min_v / 1e6:.0f} MHz")
print(f"dropped_transition_count: {dropped_before} -> {dropped_after}")
if dropped_after > dropped_before:
    print("WARNING: this transition never reached the DAC -- see ScappFeeder "
          "docstring; consider raising notify_samples.")
else:
    print("OK -- transition was rendered within the DMA chunk timing.")

assert feeder.last_error is None, feeder.last_error

if "sva" in globals() and sva is not None:
    freq_hz, amp_dbm = read_peak()
    print(f"SVA1015X peak: {freq_hz / 1e6:.3f} MHz @ {amp_dbm:.1f} dBm "
          f"(expected ~{aod.f_min_v / 1e6:.0f} MHz)")


# 6. Cleanup

Always run this before ending the session (even after an error above) -- it stops
the feeder thread and DMA transfer, closes the card, and disconnects the spectrum
analyzer. A card left streaming keeps driving the AOD.

In [ ]:
if "feeder" in globals() and feeder is not None:
    feeder.stop()
    print("Feeder stopped.")
    feeder = None

if "card" in globals() and card is not None:
    card.close()
    print("Card closed.")
    card = None

if "sva" in globals() and sva is not None:
    sva.close()
    print("Spectrum analyzer connection closed.")
    sva = None
